# 物理可信资产构建

本 Demo 依次介绍碰撞体、物性辨识、资产组装和 OpenUSD 封装 API。每个功能都是独立单元格，不调用总流程函数。


## 0. 初始化平台

只定义工作区与输出目录；每项功能的方法与参数紧邻 API 调用展示。


In [ ]:
# ruff: noqa
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("未找到 PhysReal-X 项目根目录。")
    PROJECT_ROOT = PROJECT_ROOT.parent
for import_root in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from examples._common import operation_receipt, write_summary
from physrealx import PhysRealX
from physrealx.errors import PhysRealXError

WORKSPACE = PROJECT_ROOT / ".physrealx/examples/02-physical-assets"
OUTPUT_DIR = PROJECT_ROOT / "examples/physical_asset_building/output"
px = PhysRealX.open(WORKSPACE)
receipts = {}

## 1. 生成碰撞几何：`assets.build_collision`

**用途**：为每个 link 的视觉 Mesh 生成可用于物理引擎的近似凸碰撞体。

**输出**：`CollisionGeometry/1` 和体积、凸包数、几何偏差等报告。


In [ ]:
visual_meshes = []  # 例如 [("drawer", drawer_visual_mesh_artifact)]
collision_operations = []
if not visual_meshes:
    print("[BLOCKED] 请在本单元格为每个 link 绑定 VisualMesh/1。")
else:
    for link, visual_mesh in visual_meshes:
        try:
            operation = px.assets.build_collision(
                visual_mesh,
                method="coacd",
                config={"source_unit": "m", "link": link},
            )
            collision_operations.append(operation)
            print(link, operation.metrics)
        except PhysRealXError as error:
            print(f"[BLOCKED] {link}: {error}")
receipts["collision_operations"] = [operation_receipt(item) for item in collision_operations]

## 2. 辨识关节物性：`assets.calibrate_physics`

**用途**：从实测的时间、位置、速度、控制量和力/力矩估计惯量、摩擦与阻尼。

**质量门禁**：激励充分性、可辨识性、多初值收敛及留出轨迹误差必须通过。


In [ ]:
system_identification_evidence = None  # 替换为实测 SystemIdentificationEvidence/1 VersionedRef
physics_operation = None
if system_identification_evidence is None:
    print("[BLOCKED] 请在本单元格绑定实测系统辨识证据。")
else:
    try:
        physics_operation = px.assets.calibrate_physics(
            system_identification_evidence,
            method="physrealx_mujoco_sysid_adapter",
            config={"multi_start": 8},
        )
        receipts["physics_calibration"] = operation_receipt(physics_operation)
        print(json.dumps(physics_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 3. 组装可训练资产：`recipes.execute`

**用途**：按显式配方合并视觉几何、碰撞几何、关节、link 物性和单关节辨识结果。

**输出**：闭包完整的 `AssetAssemblyBundle/1`；单关节辨识结果不会被冒充为整体资产物性。


In [ ]:
assembly_recipe = None  # 替换为绑定几何、关节与物性的 RecipeExecutionSpec
assembly_execution = None
if assembly_recipe is None:
    print("[BLOCKED] 请在本单元格构造 asset.assemble 配方。")
else:
    try:
        assembly_execution = px.recipes.execute(assembly_recipe)
        receipts["assembly_execution"] = assembly_execution.execution
        print(assembly_execution.execution)
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 4. 封装 OpenUSD 资产：`assets.package`

**用途**：将通过组装检查的资产写入自包含 USD，包括视觉层、碰撞层、关节、材质和物性。

**输出**：`UsdAssetBundle/1`、封装清单与往返校验报告。


In [ ]:
asset_assembly = None  # 替换为已审核 AssetAssemblyBundle/1 VersionedRef
package_operation = None
if asset_assembly is None:
    print("[BLOCKED] 请先审核并注册 AssetAssemblyBundle/1。")
else:
    try:
        package_operation = px.assets.package(
            asset_assembly, method="openusd", config={"up_axis": "Z"}
        )
        receipts["usd_package"] = operation_receipt(package_operation)
        print(json.dumps(package_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 5. 保存回执

只记录真实发布的操作结果，缺少证据的阶段保持阻断。


In [ ]:
report = write_summary(
    OUTPUT_DIR / "physical-asset-report.json",
    {"module": "物理可信资产构建", "receipts": receipts},
)
print(json.dumps(report, ensure_ascii=False, indent=2, default=str))